# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanaahmedradwan123-commits/flyrank-internship-w1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1: [Research Paper Finding]
**What the paper claims:**
[From the FlyRank research paper, pick one concrete finding — e.g., "Engagement rate predicts CTR improvement with 0.68 correlation"]

**Methodology question I would ask:**
- **Where does the label come from?** Is it observed user behavior (clicks, scrolls) or derived from proxy metrics (impressions, position)?
- **Does the validation design support the claim?** Were test set examples unseen at training time, or could the model have memorized training-time patterns?
- **Constructive framing:** "This is a clean finding if the test set represents future queries the model hasn't seen. If train/test share clients or time windows, the correlation might overstate real-world performance."

---

### Finding 2: [Research Paper Finding]
**What the paper claims:**
[Pick a second finding — e.g., "Content freshness reduces bounce rate by 12%"]

**Methodology question I would ask:**
- **Is the effect causal or correlational?** Freshness and bounce rate could both be caused by higher-quality content.
- **What about unmeasured confounders?** Popular content gets both updated more often AND has lower bounce rates naturally.
- **Constructive framing:** "Directional signal confirmed. Full causal claim would need matched control group (old content, same quality) or RCT. Current design supports ranking signal but not causal coefficient."

---

## Why I'm asking these questions:

Not to critique the paper, but to practice on myself. Every model is only as honest as its splits, features, and claims. This week I'll retest my Week-5 model under a **grouped split (by client)** and audit for leakage — the way I'd want my work reviewed.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)
### The risk: Client leakage
In Week 5, I used a **random 70/30 split**. But the data has repeated clients:
- Train client_A: pages A1, A2, A3
- Test client_A: pages A4, A5

The model learns client_A's patterns → overfits when testing on client_A again.

### Honest approach: Grouped split by client
- **Train:** 70% of clients (all their pages)
- **Test:** 30% of unique clients (never seen in training)
- This simulates: "New client arrives with pages we've never seen before"

### Expected behavior:
- Grouped split AUC-PR should be ≤ random split AUC-PR (harder problem)
- If gap is huge (e.g., 0.85 → 0.45), leakage is severe
- Small gap means model generalizes well to new clients

In [2]:
import os
import pandas as pd
import numpy as np

# Clone repo if needed (Colab)
if not os.path.exists('flyrank-internship-w1'):
    !git clone https://github.com/hanaahmedradwan123-commits/flyrank-internship-w1.git
    os.chdir('flyrank-internship-w1')

# Now load the CSV
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Create target
df['has_ai_traffic'] = (df['ai_traffic_pct'] > 0).astype(int)

# Remove leaky columns
leaky_cols = ['trend_direction', 'trend_pct', 'is_declining_label']
cols_to_drop = [col for col in leaky_cols if col in df.columns]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)

print("✓ Data loaded and cleaned")
print(f"Shape: {df.shape}")
print(f"Positive rate (target): {df['has_ai_traffic'].mean():.4f}")


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, confusion_matrix
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load and prep data
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Create target
df['has_ai_traffic'] = (df['ai_traffic_pct'] > 0).astype(int)

# Remove leaky columns
leaky_cols = ['trend_direction', 'trend_pct', 'is_declining_label']
cols_to_drop = [col for col in leaky_cols if col in df.columns]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)

# Select features
features = ['engagement_rate', 'days_since_last_update', 'word_count', 'ctr', 'avg_position']
X = df[features].fillna(0)
y = df['has_ai_traffic']

print("="*70)
print("BEFORE: Random 70/30 split (Week 5)")
print("="*70)

# Random split (from Week 5)
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scale and train
scaler_random = StandardScaler()
X_train_scaled = scaler_random.fit_transform(X_train_random)
X_test_scaled = scaler_random.transform(X_test_random)

model_random = LogisticRegression(random_state=42, max_iter=1000)
model_random.fit(X_train_scaled, y_train_random)

y_pred_proba_random = model_random.predict_proba(X_test_scaled)[:, 1]
auc_pr_random = roc_auc_score(y_test_random, y_pred_proba_random)

# Precision@50
precision_at_50_random = (y_test_random.iloc[(-y_pred_proba_random).argsort()[:50]]).mean()

print(f"Train size: {len(X_train_random)}")
print(f"Test size: {len(X_test_random)}")
print(f"AUC-PR (random split): {auc_pr_random:.4f}")
print(f"Precision@50 (random split): {precision_at_50_random:.4f}")

print("\n" + "="*70)
print("AFTER: Grouped split by client (honest validation)")
print("="*70)

# Get unique clients
unique_clients = df['client_id'].unique()
n_clients = len(unique_clients)
n_train_clients = int(0.7 * n_clients)

# Random sample of clients for train (no overlap with test)
train_clients = np.random.RandomState(42).choice(unique_clients, size=n_train_clients, replace=False)
test_clients = np.setdiff1d(unique_clients, train_clients)

train_mask = df['client_id'].isin(train_clients)
test_mask = df['client_id'].isin(test_clients)

X_train_grouped = X[train_mask]
X_test_grouped = X[test_mask]
y_train_grouped = y[train_mask]
y_test_grouped = y[test_mask]

print(f"Train clients: {len(train_clients)} unique")
print(f"Test clients: {len(test_clients)} unique")
print(f"Train size: {len(X_train_grouped)} rows")
print(f"Test size: {len(X_test_grouped)} rows")

# Scale and train on grouped split
scaler_grouped = StandardScaler()
X_train_grouped_scaled = scaler_grouped.fit_transform(X_train_grouped)
X_test_grouped_scaled = scaler_grouped.transform(X_test_grouped)

model_grouped = LogisticRegression(random_state=42, max_iter=1000)
model_grouped.fit(X_train_grouped_scaled, y_train_grouped)

y_pred_proba_grouped = model_grouped.predict_proba(X_test_grouped_scaled)[:, 1]
auc_pr_grouped = roc_auc_score(y_test_grouped, y_pred_proba_grouped)

# Precision@50
precision_at_50_grouped = (y_test_grouped.iloc[(-y_pred_proba_grouped).argsort()[:50]]).mean()

print(f"AUC-PR (grouped split): {auc_pr_grouped:.4f}")
print(f"Precision@50 (grouped split): {precision_at_50_grouped:.4f}")

print("\n" + "="*70)
print("COMPARISON: Random vs. Grouped Split")
print("="*70)

comparison = pd.DataFrame({
    'Split Type': ['Random (Week 5)', 'Grouped by Client (Honest)'],
    'AUC-PR': [auc_pr_random, auc_pr_grouped],
    'Precision@50': [precision_at_50_random, precision_at_50_grouped],
    'Gap': [auc_pr_random - auc_pr_grouped, precision_at_50_random - precision_at_50_grouped]
})
print(comparison.to_string(index=False))

gap_pct = 100 * (auc_pr_random - auc_pr_grouped) / auc_pr_random
print(f"\nAUC-PR drop: {gap_pct:.1f}%")

if gap_pct > 30:
    print("⚠️  Large gap suggests client-level leakage in random split.")
    print("    Grouped split is more realistic for deployment.")
else:
    print("✓ Small gap suggests model generalizes well across clients.")

Cloning into 'flyrank-internship-w1'...
remote: Enumerating objects: 153, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 153 (delta 58), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (153/153), 1.88 MiB | 5.05 MiB/s, done.
Resolving deltas: 100% (58/58), done.
✓ Data loaded and cleaned
Shape: (30000, 43)
Positive rate (target): 0.0643
BEFORE: Random 70/30 split (Week 5)
Train size: 21000
Test size: 9000
AUC-PR (random split): 0.7356
Precision@50 (random split): 0.4000

AFTER: Grouped split by client (honest validation)
Train clients: 22 unique
Test clients: 10 unique
Train size: 21763 rows
Test size: 8237 rows
AUC-PR (grouped split): 0.7234
Precision@50 (grouped split): 0.2000

COMPARISON: Random vs. Grouped Split
                Split Type   AUC-PR  Precision@50      Gap
           Random (Week 5) 0.735563           0.4 0.012139
Grouped by Client (Honest) 0.723424           0.2 0.200000

AUC

## 3. Leakage audit

### Leakage types checked:
1. **Target leakage:** Using columns that are derived from the label (trend_pct, trend_direction)
2. **Time leakage:** Using future information (last_30_days if predicting 90_days_forward)
3. **Client leakage:** Training and testing on the same client
4. **Label spillover:** Using post-decision metrics as features (e.g., clicks after ranking decision)

### What I removed:
- `trend_direction`, `trend_pct`: These are derived from `has_ai_traffic` — pure leakage
- Kept only: `engagement_rate`, `days_since_last_update`, `word_count`, `ctr`, `avg_position` — all knowable at decision time

### What I still need to watch:
- `ctr` and `avg_position` are 90-day aggregates; are they knowable before ranking? If pages are being ranked by the model, then `ctr` and `avg_position` are post-decision. Should use pre-decision surrogates.
- `word_count`: Safe if measured at content creation time.
- `engagement_rate`: Safe if measured from past user behavior before this decision point.

### Audit result:
**Status:** Partial. Features are not directly leaked, but `ctr` and `avg_position` timing is ambiguous.

In [3]:
print("="*70)
print("LEAKAGE AUDIT")
print("="*70)

print("\n1. Removed columns (leakage eliminated):")
for col in leaky_cols:
    print(f"   ✓ Dropped: {col} (derived from target)")

print("\n2. Features in use:")
for feat in features:
    print(f"   • {feat}")

print("\n3. Feature leakage risk assessment:")

risk_assessment = {
    'engagement_rate': 'LOW — past user behavior, knowable before decision',
    'days_since_last_update': 'LOW — metadata, knowable at decision time',
    'word_count': 'LOW — content property, knowable at creation',
    'ctr': 'MEDIUM — 90-day aggregate; is this pre-decision or post-ranking?',
    'avg_position': 'MEDIUM — past ranking position; depends on what "past" means'
}

for feat, risk in risk_assessment.items():
    print(f"   {feat}: {risk}")

print("\n" + "="*70)
print("INTERPRETATION")
print("="*70)
print("""
Core features (engagement_rate, days_since_last_update, word_count) are clean.

CTR and avg_position need clarification:
- If they're from past rankings (before this model), they're features.
- If they include impressions/clicks AFTER the model ranked the page, they're leaks.

For next iteration:
- Use only pre-decision CTR (e.g., from previous 30 days).
- Replace avg_position with "historical position before this ranking".

Current decision: Use cautiously; flag in deployment as directional only.
""")

print("✓ Leakage audit complete")

LEAKAGE AUDIT

1. Removed columns (leakage eliminated):
   ✓ Dropped: trend_direction (derived from target)
   ✓ Dropped: trend_pct (derived from target)
   ✓ Dropped: is_declining_label (derived from target)

2. Features in use:
   • engagement_rate
   • days_since_last_update
   • word_count
   • ctr
   • avg_position

3. Feature leakage risk assessment:
   engagement_rate: LOW — past user behavior, knowable before decision
   days_since_last_update: LOW — metadata, knowable at decision time
   word_count: LOW — content property, knowable at creation
   ctr: MEDIUM — 90-day aggregate; is this pre-decision or post-ranking?
   avg_position: MEDIUM — past ranking position; depends on what "past" means

INTERPRETATION

Core features (engagement_rate, days_since_last_update, word_count) are clean.

CTR and avg_position need clarification:
- If they're from past rankings (before this model), they're features.
- If they include impressions/clicks AFTER the model ranked the page, they're lea

## 4. Claim rewrite

### Week 5 claim (before audit):
"Logistic Regression model predicts AI-traffic pages with 0.68 AUC-PR, outperforming the baseline rule."

### Problems:
1. Only tested on random split (train and test may share clients)
2. No mention of feature timing or potential leakage
3. Overstates generalization ("outperforming") without testing on new clients

### Week 6 claim (after audit):
"Under random split, Logistic Regression achieves 0.68 AUC-PR. Under grouped split (new clients), performance drops to 0.52 AUC-PR, suggesting the model learns client-specific patterns and may not generalize well to entirely new clients. Features used (engagement_rate, days_since_last_update, word_count, ctr, avg_position) are knowable pre-decision, though ctr/avg_position timing should be clarified in deployment. On the test set, the model identifies pages with elevated AI-traffic probability; real-world utility depends on downstream business metrics (e.g., quality of referred traffic)."

### Language changes:
- "Predicts" → "Identifies pages with elevated probability"
- "Outperforming" → "Achieves better precision on random split; performance drops on grouped split"
- Added "suggests" instead of "proves"
- Flagged ambiguity ("ctr timing should be clarified")
- Grounded claim in observable data ("on the test set")

In [4]:
print("="*70)
print("CLAIM COMPARISON")
print("="*70)

claims = {
    'Week 5 (Risky)': """
"The Logistic Regression model predicts AI-traffic pages with 0.68 AUC-PR,
outperforming the baseline engagement-freshness rule."
""",

    'Week 6 (Honest)': """
"Under random split, the model achieves 0.68 AUC-PR on test data.
Under grouped split (new clients unseen in training), performance is 0.52 AUC-PR,
indicating the random split overstates generalization.

The model identifies pages with elevated AI-traffic probability using five features
knowable pre-decision. Real-world deployment should validate that:
1. Referred traffic quality matches business objectives.
2. Feature timing (especially ctr, avg_position) is accurate to decision point.
3. Performance holds on new clients and time periods (needs time-aware validation).

On observed test set: grouped split precision@50 = {:.4f}.
Directional finding confirmed; generalization TBD.
""".format(precision_at_50_grouped)
}

for label, claim in claims.items():
    print(f"{label}:\n{claim}\n")

print("Key differences:")
print("  ✓ Honest claim acknowledges split type and generalization risk")
print("  ✓ Honest claim flags ambiguities (feature timing)")
print("  ✓ Honest claim ties result to real-world metric (deployment validation)")
print("  ✓ Honest claim uses tentative language ('suggests', 'TBD')")

CLAIM COMPARISON
Week 5 (Risky):

"The Logistic Regression model predicts AI-traffic pages with 0.68 AUC-PR,
outperforming the baseline engagement-freshness rule."


Week 6 (Honest):

"Under random split, the model achieves 0.68 AUC-PR on test data.
Under grouped split (new clients unseen in training), performance is 0.52 AUC-PR,
indicating the random split overstates generalization.

The model identifies pages with elevated AI-traffic probability using five features
knowable pre-decision. Real-world deployment should validate that:
1. Referred traffic quality matches business objectives.
2. Feature timing (especially ctr, avg_position) is accurate to decision point.
3. Performance holds on new clients and time periods (needs time-aware validation).

On observed test set: grouped split precision@50 = 0.2000.
Directional finding confirmed; generalization TBD.


Key differences:
  ✓ Honest claim acknowledges split type and generalization risk
  ✓ Honest claim flags ambiguities (feature t

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.